# 04 · Collect everything for the paper

Merges `results/cnn/paper_cnn.json` and `results/rag/paper_rag.json` into `results/paper_numbers.json`, using the
schema of `IEEE-Paper/scripts/mock_data.json` (blocks with `"is_mock": false`), and writes `results/RESULTS.md`.
It does **not** touch `IEEE-Paper/`: copying the numbers into the paper is a separate step.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

from common import config, env

parts = {}
for name, path in (("cnn", config.CNN_RESULTS / "paper_cnn.json"), ("rag", config.RAG_RESULTS / "paper_rag.json")):
    if path.exists():
        parts[name] = json.loads(path.read_text())
    else:
        print(f"missing {path.relative_to(config.ROOT)} - run the {name.upper()} notebooks first; skipping")

out = {"_README": "Real results from compare/ (is_mock=false). Generated by 04_export/04_paper_numbers.ipynb.",
       "meta": env.run_meta()}
for p in parts.values():
    out.update(p)
env.write_json(config.RESULTS_DIR / "paper_numbers.json", out)

md = ["# PlantIQ comparison results", "", f"Generated {out['meta']['timestamp']} at commit {out['meta']['git_commit']}.",
      "Every number below is traceable to a JSON file in `results/`.", ""]
for f in ("RESULTS_cnn.md", "RESULTS_rag.md"):
    p = config.RESULTS_DIR / f
    if p.exists():
        md += [p.read_text(encoding="utf-8"), ""]
audit = config.AUDIT_DIR / "audit_report.md"
if audit.exists():
    md += ["## Dataset audit", "", audit.read_text(encoding="utf-8")]
(config.RESULTS_DIR / "RESULTS.md").write_text("\n".join(md), encoding="utf-8")
print("wrote", config.RESULTS_DIR / "paper_numbers.json")
print("wrote", config.RESULTS_DIR / "RESULTS.md")
print("blocks:", [k for k in out if k not in ("_README", "meta")])